> 在线讲义：[第 15 章 · RoPE：要分数只看相对距离，就把 q、k 按位置转一个角度](https://wikieden.github.io/beyond-transformer/lessons/15-rope.html) · 本书《超越 Transformer》

# 第 15 章 · RoPE：要分数只看相对距离，就把 q、k 按位置转一个角度

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/15-rope.html) · 仓库里 `beyond-transformer/notes/lessons/15-rope.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 二维 | R_mθ q · R_nθ k | `score_2d(m, n)` | 差 1 都是 0.414105 |
| 第三步 频率 | 10000^{−2i/d} | `freqs(8)`、`wavelengths()` | [1, 0.1, 0.01, 0.001]；6.28 … 54,410 |
| 第四步 高效实现 | 式 (34) | `rope(x, 3)` vs `rope_matrix(x, 3)` | 完全一样 |
| 第五步 衰减 | |Σe^{irθ}| | `decay_curve()` | 1 → 0.1125 |

d 维拆成 d/2 个平面、每个平面一个频率 θ_i = 10000^{−2i/d}；按元素乘 cos、sin 的高效写法；远距离衰减。
a1 的 rope（PyTorch 模块）留给自己写；五种位置编码的外推对比是里程碑 T-M3。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `rot2`**：小工具。

In [ ]:
def rot2(v, angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([c * v[0] - s * v[1], s * v[0] + c * v[1]])

In [ ]:
Q = np.array([1.0, 0.5])
K = np.array([0.8, -0.3])
THETA = 0.3

**函数 `score_2d`**：位置 m 的 q 转 mθ，位置 n 的 k 转 nθ，再点积。

In [ ]:
def score_2d(m, n, q=Q, k=K, theta=THETA):
    """位置 m 的 q 转 mθ，位置 n 的 k 转 nθ，再点积。"""
    return float(rot2(q, m * theta) @ rot2(k, n * theta))

**函数 `freqs`**：小工具。

In [ ]:
def freqs(d=8, base=10000.0):
    return base ** (-np.arange(0, d, 2) / d)

**函数 `rope`**：高效写法（RoFormer §3.4.2 式 34）：x ⊙ cos + rotate_half(x) ⊙ sin。相邻两维 (x₀, x₁)、(x₂, x₃)… 各成一个平面。

In [ ]:
def rope(x, pos, base=10000.0):
    """高效写法（RoFormer §3.4.2 式 34）：x ⊙ cos + rotate_half(x) ⊙ sin。相邻两维 (x₀, x₁)、(x₂, x₃)… 各成一个平面。"""
    d = len(x)
    th = np.repeat(freqs(d, base), 2) * pos
    xr = np.empty_like(x)
    xr[0::2], xr[1::2] = -x[1::2], x[0::2]
    return x * np.cos(th) + xr * np.sin(th)

**函数 `rope_matrix`**：笨写法：拼一个分块对角的旋转矩阵再乘，用来核对。

In [ ]:
def rope_matrix(x, pos, base=10000.0):
    """笨写法：拼一个分块对角的旋转矩阵再乘，用来核对。"""
    d = len(x)
    R = np.zeros((d, d))
    for i, t in enumerate(freqs(d, base)):
        c, s = np.cos(pos * t), np.sin(pos * t)
        R[2 * i : 2 * i + 2, 2 * i : 2 * i + 2] = [[c, -s], [s, c]]
    return R @ x

**函数 `decay_curve`**：RoFormer §3.4.3：q、k 全是 1 时，(1/(d/2))·|Σ_i e^{i·rθ_i}| 随相对距离 r 的变化（上界的直观版）。

In [ ]:
def decay_curve(d=128, base=10000.0, dists=(0, 1, 10, 100, 1000, 10000)):
    """RoFormer §3.4.3：q、k 全是 1 时，(1/(d/2))·|Σ_i e^{i·rθ_i}| 随相对距离 r 的变化（上界的直观版）。"""
    th = freqs(d, base)
    return {r: float(np.abs(np.exp(1j * r * th).sum()) / len(th)) for r in dists}

**函数 `wavelengths`**：每个平面转一圈要多少个位置：2π / θ_i。

In [ ]:
def wavelengths(d=128, base=10000.0):
    """每个平面转一圈要多少个位置：2π / θ_i。"""
    return 2 * np.pi / freqs(d, base)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 二维：只依赖 m − n")
for m, n in ((1, 0), (5, 4), (10, 9), (3, 0), (7, 4)):
    print(f"  m = {m:2d}、n = {n}（差 {m - n}）：q_m·k_n = {score_2d(m, n):.6f}")
print("\n第二步 d 维：每两维一个平面，频率越来越慢")
print(f"  d = 8：θ = {np.round(freqs(8), 6).tolist()}")
wl = wavelengths()
print(f"  d = 128：转一圈要 {wl[0]:.2f} 个位置（最快）……{wl[-1]:,.0f} 个位置（最慢）")

In [ ]:
print("\n第三步 高效写法和旋转矩阵一样")
x = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0])
a, b = rope(x, 3), rope_matrix(x, 3)
print(f"  位置 3：{np.round(a, 4).tolist()}；和矩阵写法最多差 {np.abs(a - b).max():.1e}")

In [ ]:
print("\n第四步 远距离衰减（q、k 全 1 时的平均相干程度）")
for r, v in decay_curve().items():
    print(f"  相对距离 {r:5d}：{v:.4f}")

## 3. 代码里才看得到的

- **`np.repeat` 决定了配对方式**：重复成 [θ₀, θ₀, θ₁, θ₁, …] 就是相邻配对；Hugging Face 用 `cat([freqs, freqs])` 得到 [θ₀, θ₁, …, θ₀, θ₁, …]，是前后一半配对。
- **测试里还核对了一般的 d 维**：随机的 q、k，位置 (7, 4) 和 (13, 10) 的点积一样；这是“只依赖 m − n”的 d 维版本。在真实模型上比较各种位置编码的长度外推是里程碑 T-M3。

## 4. 自己改一改

- 把 `decay_curve` 的底数从 10000 改成 500000（LLaMA 3 用的值），衰减变慢还是变快？
- 把 `score_2d` 的 θ 改成 2π，会发生什么？为什么频率不能太大？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 两人同速转圈跑，夹角只看谁先出发；q 转 mθ、k 转 nθ，点积只剩 m 减 n。
> 两两一组成平面，频率从快到慢，一万的负 2i/d 次方。
> 不乘大矩阵：x 乘 cos，加转半个乘 sin；配对方式要和权重配套。
> 距离越远相位越乱，分数天生偏低。